# Galaxy-count metrics: GalaxyCountsMetricExtended vs DepthLimitedNumGalMetric on the dust-footprint variants

- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-19
- Kernel: conda_py313_opsim53
- Context: SCOC footprint-shrinking study + DESC static-probes metrics. The goal of the series `10_DESCMAFDEPTHANDGALCOUNT` is to understand three `rubin_sim` MAF metrics in detail in order to improve them later (PSF handling, weak-lensing metric).
- This notebook: **02** of the series - the two galaxy-count metrics **`GalaxyCountsMetricExtended`** and **`DepthLimitedNumGalMetric`**: how they are computed, where they differ, and their Healpix maps, histograms and consecutive differences for each dust-threshold footprint variant of the v5.3.6 simulations.
- Companion notebook: `01_compareExgalM5withCuts.ipynb` (`ExgalM5WithCuts`, the footprint/depth definition reused inside `DepthLimitedNumGalMetric`). Source code of the metrics: `00_DepthsAndCountsMetrics.ipynb`. Earlier demo of `GalaxyCountsMetricExtended` on the baseline: `../06_MAF_DESC_TaskF/07_galaxycounts.ipynb`.
- Presentation model: `../07_variateEVmV/01_FOMNv_HealpixDiff_ShrinkFPDust.ipynb`
- OpSim simulations analyzed (`/Users/dagoret/DATA/OpSim/`):
  - `shrink_fp_dust_0.050_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.080_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.120_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.150_v5.3.6_10yrs.db`
  - `shrink_fp_dust_0.199_v5.3.6_10yrs.db`
  - `baseline_v5.3.6_11yrs.db` (E(B-V) threshold 0.200)
  - `shrink_fp_dust_0.250_v5.3.6_10yrs.db`


## Notebook overview

**The physical model shared by both metrics** (Awan et al. 2016). For one pixel:
1. the coadded, dust-corrected i-band depth `coaddm5` is computed with `ExgalM5` (coadd of the point-source `fiveSigmaDepth` of the visits, minus the extinction `A_i = ax1['i'] * E(B-V)`);
2. the galaxy differential counts are `dN/dm = sum over redshift bins of 10**(a*m + b)` per deg² per magnitude (power laws fitted on the Padilla et al. mock catalogs, constants in `constants_for_pipeline.py`), multiplied by `normalization_constant` so that the total matches the CFHTLS counts at i < 25.5;
3. the counts are multiplied by a completeness `0.5 * erfc(m - coaddm5)` (50% at `m = coaddm5`) and integrated over the magnitude up to `upper_mag_limit`;
4. the result per deg² is scaled to one Healpix pixel (`41253 / (12 nside²)` deg²).

**Where the two metrics differ** (from the `rubin_sim` source, main branch):

| | `GalaxyCountsMetricExtended` | `DepthLimitedNumGalMetric` |
|---|---|---|
| Role | galaxies per pixel from the coadded depth | same, restricted to the extragalactic footprint and to a depth-limited sample |
| Coadded depth | `ExgalM5` on the visits of `filter_band` | same: it runs an internal `GalaxyCountsMetricExtended` |
| Footprint cuts | **none**: any pixel with at least one visit in the band gets a value (0 if too shallow), masked only if no visit in the band | **`ExgalM5WithCuts`** inside: `E(B-V) < lim_ebv` (0.2 by default; here set run by run to the E(B-V) threshold of the run, see Section 2), all `nfilters_needed` (6) bands observed, coadded i depth >= `lim_mag_i_ptsrc` (26.0); otherwise masked |
| Upper magnitude limit of the integral | `upper_mag_limit = 32` (in practice the completeness term ends the integral, about 1 mag beyond `coaddm5`) | `lim_mag_i_ptsrc - 0.7 = 25.3`, a **hard cut that does not depend on the pixel depth** |
| SQL constraint | a band constraint is optional (the metric selects `filter_band` itself) | **no band constraint** (all bands are needed for the coverage cut) |
| Default `nside` | 128 | 128 in the signature (the docstring says 256); must match the slicer |

The `- 0.7` in `DepthLimitedNumGalMetric` is a fixed offset between the point-source and the extended-source limiting magnitude (comment in the source: galaxies are about twice as large as the seeing). It is the only place where anything like a PSF/size effect enters the two metrics, and it does not use the actual seeing.

**What this notebook does.**
1. *Single-pixel experiment* (Sections 4-5, no simulation needed): the real metric objects are run on a synthetic pixel of given coadded depth, to see the count as a function of depth for both metrics, the part of the integrand that the 25.3 cut removes, and how sensitive the count is to the 0.7 mag offset.
2. Both metrics are run on the 7 simulations on the same visits (same query, same slicer, same dust map), at `nside = 128`, i band, all redshifts, first 10 years, non-DDF visits. The E(B-V) cut `lim_ebv` of `DepthLimitedNumGalMetric` is set **run by run** to the threshold that defines the WFD (Wide Fast Deep) footprint of the run (`EBV_CUT_MODE = 'run'`, Section 2); `GalaxyCountsMetricExtended` has no E(B-V) cut, so its maps do not depend on this choice. Maps are cached as `.npz` files.
3. Maps, area-weighted histograms and consecutive-threshold difference maps for each metric, in the format of the 07_variateEVmV notebook. For counts, a pixel that is masked counts as **0 galaxies**, so a difference map also contains the galaxies gained or lost with the footprint; the table splits the total difference into *common pixels*, *gained pixels* and *lost pixels*.
4. The total counts of `GalaxyCountsMetricExtended` are split into a **footprint effect** (restricting to the `DepthLimitedNumGalMetric` footprint) and a **truncation effect** (the 25.3 magnitude cut).


## 1. Imports

In [ ]:
import os
import inspect
from os.path import join, isfile

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maf_maps
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.maf_contrib.lss_obs_strategy.galaxy_counts_metric_extended import (
    GalaxyCountsMetricExtended,
)
from rubin_sim.maf.maf_contrib.depth_limited_num_gal_metric import DepthLimitedNumGalMetric
from rubin_sim.maf.maf_contrib.lss_obs_strategy.constants_for_pipeline import (
    power_law_const_a,
    power_law_const_b,
    normalization_constant,
)

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

In [ ]:
NB_TAG = "GALCOUNTS"
data_dir = f"data_02_{NB_TAG}"
figs_dir = f"figs_02_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

resultsDb = maf.db.ResultsDb(out_dir=data_dir)

In [ ]:
# Directory holding the OpSim databases
OPSIM_DIR = "/Users/dagoret/DATA/OpSim"

# (run_name, E(B-V) threshold that defines the footprint of the run), sorted by increasing threshold
RUNS_INFO = [
    ("shrink_fp_dust_0.050_v5.3.6_10yrs", 0.050),
    ("shrink_fp_dust_0.080_v5.3.6_10yrs", 0.080),
    ("shrink_fp_dust_0.120_v5.3.6_10yrs", 0.120),
    ("shrink_fp_dust_0.150_v5.3.6_10yrs", 0.150),
    ("shrink_fp_dust_0.199_v5.3.6_10yrs", 0.199),
    ("baseline_v5.3.6_11yrs", 0.200),
    ("shrink_fp_dust_0.250_v5.3.6_10yrs", 0.250),
]
RUN_NAMES = [r for r, _ in RUNS_INFO]
DUST_THRESH = dict(RUNS_INFO)
REF_RUN = "baseline_v5.3.6_11yrs"

# Consecutive-threshold pairs (later run minus earlier run):
# 0.080-0.050, 0.120-0.080, 0.150-0.120, 0.199-0.150, baseline-0.199, 0.250-baseline
PAIRS = [(RUN_NAMES[i + 1], RUN_NAMES[i]) for i in range(len(RUN_NAMES) - 1)]


def get_db_path(run_name):
    # Look in OPSIM_DIR, then in OPSIM_DIR/sim_baseline (where the baseline was stored in the 06/07 notebooks)
    fname = run_name + ".db"
    candidates = [join(OPSIM_DIR, fname), join(OPSIM_DIR, "sim_baseline", fname)]
    for c in candidates:
        if isfile(c):
            return c
    raise FileNotFoundError(f"OpSim db not found for run {run_name}. Tried: {candidates}")


for run_name, dust in RUNS_INFO:
    try:
        path = get_db_path(run_name)
    except FileNotFoundError:
        path = "NOT FOUND (only a problem if there is no cached map, see section 7)"
    print(f"{run_name:40s} E(B-V) < {dust:.3f}  ->  {path}")

In [ ]:
BAND = "i"
NSIDE = (
    128  # default of both metrics; must match the slicer nside (used for the sq-deg -> pixel scale factor)
)
MAX_NIGHT = 10 * 365.25 + 0.5  # keep the first 10 years of every run

# No band constraint: DepthLimitedNumGalMetric needs the visits of all bands for its coverage cut.
# GalaxyCountsMetricExtended selects filter_band itself, so it sees the same visits.
SQL_CONSTRAINT = "night <= %s and scheduler_note not like 'DD%%'" % MAX_NIGHT
INFO_LABEL = "GalCounts 10yr nonDD"

# Parameters of the two metrics: the library defaults, written explicitly so they can be varied
GC_KWARGS = dict(
    upper_mag_limit=32.0,
    include_dust_extinction=True,
    redshift_bin="all",
    cfht_ls_counts=False,
    normalized_mock_catalog_counts=True,
)
DL_KWARGS = dict(
    redshift_bin="all", nfilters_needed=6, lim_mag_i_ptsrc=26.0
)  # lim_ebv is set run by run, see metric_ebv_cut() below

gc_up = GC_KWARGS["upper_mag_limit"]
dl_pt = DL_KWARGS["lim_mag_i_ptsrc"]
# (the E(B-V) cut of DepthLimitedNumGalMetric is not fixed here: see EBV_CUT_MODE below)
dl_nf = DL_KWARGS["nfilters_needed"]

# E(B-V) cut given to DepthLimitedNumGalMetric (lim_ebv), see metric_ebv_cut():
#   'run'   -> E(B-V) threshold that defines the WFD footprint of each run (DUST_THRESH), adapted run by run
#   'fixed' -> the same value FIXED_LIM_EBV for all runs (previous behaviour of this notebook)
# GalaxyCountsMetricExtended has no E(B-V) cut: its maps do not depend on this choice.
EBV_CUT_MODE = "run"
FIXED_LIM_EBV = 0.2


def metric_ebv_cut(run_name):
    # E(B-V) cut given to DepthLimitedNumGalMetric for this run
    if EBV_CUT_MODE == "run":
        return DUST_THRESH[run_name]
    if EBV_CUT_MODE == "fixed":
        return FIXED_LIM_EBV
    raise ValueError(f"EBV_CUT_MODE must be 'run' or 'fixed', got {EBV_CUT_MODE!r}")


# Cache tags. GalaxyCountsMetricExtended (GC): deliberately the same tag as in the previous version of this
# notebook ('dlebv0.20' is a historical leftover, not a parameter), so that the GC maps already cached are reused.
GC_CONFIG_TAG = f"{BAND}_nside{NSIDE}_gcup{gc_up:.1f}_dlpt{dl_pt:.1f}_dlebv0.20_dlnf{dl_nf}"


def dl_config_tag(run_name):
    # DepthLimitedNumGalMetric (DL): the tag contains the lim_ebv actually used, so a map computed with another cut is never reloaded
    return f"{BAND}_nside{NSIDE}_dlpt{dl_pt:.1f}_dlebv{metric_ebv_cut(run_name):.3f}_dlnf{dl_nf}"


FORCE_RECOMPUTE = False  # True -> ignore the cached .npz maps and rerun MAF

pix_area = hp.nside2pixarea(NSIDE, degrees=True)
print(f"nside={NSIDE} -> npix={hp.nside2npix(NSIDE)}, pixel area = {pix_area:.4f} deg^2")
print("SQL constraint:", SQL_CONSTRAINT)
print(f"E(B-V) cut given to DepthLimitedNumGalMetric (mode '{EBV_CUT_MODE}'):")
for run_name, _ in RUNS_INFO:
    print(
        f"  {run_name:40s} lim_ebv = {metric_ebv_cut(run_name):.3f}   DL cache tag: {dl_config_tag(run_name)}"
    )
print("GC cache tag  :", GC_CONFIG_TAG)

## 3. The metrics: source code and galaxy-model constants

The printed constants let you check, on your installation, the redshift bins used by `redshift_bin='all'`: in the main-branch source that was read to write this notebook, the dictionaries contain **11** entries (the last one is `4.0<z<15.0`) and the key of the third bin has a **leading space** (`' 0.37<z<0.66'`). If this is also the case here, then (i) `redshift_bin='all'` sums 11 power laws, i.e. it goes beyond the `0<z<4` written in the docstrings, and (ii) asking for `'0.37<z<0.66'` (as written in the docstring) is not a valid key and falls back to `'all'` with a warning.

In [ ]:
print(inspect.getsource(GalaxyCountsMetricExtended._gal_count))
print("-" * 80)
print(inspect.getsource(GalaxyCountsMetricExtended.run))
print("-" * 80)
print(inspect.getsource(DepthLimitedNumGalMetric.__init__))
print("-" * 80)
print(inspect.getsource(DepthLimitedNumGalMetric.run))

In [ ]:
print("normalization_constant (mock catalog -> CFHTLS i<25.5):", normalization_constant)
print("number of redshift bins in the power-law dictionaries  :", len(power_law_const_a))
for key in power_law_const_a:
    print(f"{key!r:16s} a = {power_law_const_a[key]:6.3f}   b = {power_law_const_b[key]:7.3f}")

### 3.1 Implied redshift distribution N(z)

`GalaxyCountsMetricExtended` (and therefore `DepthLimitedNumGalMetric`, which calls it internally) does assume a galaxy redshift distribution when `redshift_bin='all'` (the default, used everywhere in this notebook): the differential magnitude counts of each of the redshift bins printed above are **summed**, each bin being a separate power law `dN/dm = 10**(a_z*m + b_z)` fitted on the Padilla et al. mock catalogs (Awan et al. 2016). This is not an explicit `N(z)` curve stored anywhere in `rubin_sim`, but it is fully equivalent to one: integrating each bin's power law over magnitude (with the same `erfc` completeness, up to a chosen depth) gives the number of galaxies **per redshift bin**, i.e. a binned `N(z)`.

The cell below reproduces this from the same `power_law_const_a` / `power_law_const_b` / `normalization_constant` already imported (same formula as `GalaxyCountsMetricExtended._gal_count`, band correction = 0 for `BAND = 'i'`), for a few reference coadded depths.

In [ ]:
import scipy.integrate as sci
import scipy.special as scs


def parse_zbin(key):
    # "0.<z<0.15", " 0.37<z<0.66", "4.0<z<15.0" -> (0.0, 0.15)
    lo, _, hi = key.strip().partition("<z<")
    return float(lo), float(hi)


zbin_keys = list(power_law_const_a.keys())
z_lo = np.array([parse_zbin(k)[0] for k in zbin_keys])
z_hi = np.array([parse_zbin(k)[1] for k in zbin_keys])
order = np.argsort(0.5 * (z_lo + z_hi))
zbin_keys = [zbin_keys[i] for i in order]
z_lo, z_hi = z_lo[order], z_hi[order]
z_center = 0.5 * (z_lo + z_hi)

BAND_CORRECTION = 0.0  # BAND == "i" (see GalaxyCountsMetricExtended._gal_count)


def gal_count_zbin(m, coaddm5, key):
    dn_gal = 10.0 ** (power_law_const_a[key] * (m + BAND_CORRECTION) + power_law_const_b[key])
    completeness = 0.5 * scs.erfc(m - coaddm5)
    return dn_gal * completeness


def counts_per_zbin(coaddm5, upper_mag_limit=GC_KWARGS["upper_mag_limit"], normalize=True):
    # galaxies / deg^2 in each redshift bin, same integral and normalization as GalaxyCountsMetricExtended
    n = np.array(
        [sci.quad(gal_count_zbin, -np.inf, upper_mag_limit, args=(coaddm5, key))[0] for key in zbin_keys]
    )
    return n * normalization_constant if normalize else n


DEPTHS_NZ = [dl_pt - 0.7, 26.2, 26.8]  # DepthLimited hard cut (25.3), single-pixel demo, ~10yr i-band coadd
colors_nz = plt.cm.viridis(np.linspace(0.15, 0.9, len(DEPTHS_NZ)))

fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for m5, c in zip(DEPTHS_NZ, colors_nz):
    n_bin = counts_per_zbin(m5)
    axs[0].bar(
        z_lo,
        n_bin / (z_hi - z_lo),
        width=z_hi - z_lo,
        align="edge",
        color=c,
        edgecolor=c,
        alpha=0.45,
        label=f"coadded depth = {m5:.2f}",
    )
    axs[1].plot(z_center, np.cumsum(n_bin) / n_bin.sum(), "-o", color=c, label=f"coadded depth = {m5:.2f}")
axs[0].set_xlim(0.0, 4.5)
axs[0].set_xlabel("redshift z")
axs[0].set_ylabel("dN/dz [galaxies / deg² / unit z]")
axs[0].set_title("Implied N(z) (last bin extends to z=15, not shown in full)")
axs[0].legend(fontsize=8)
axs[0].grid(alpha=0.3)
axs[1].set_xlim(0.0, 4.5)
axs[1].axhline(0.5, color="gray", ls="--", linewidth=0.8)
axs[1].set_xlabel("redshift z")
axs[1].set_ylabel("cumulative fraction of counted galaxies")
axs[1].set_title("Cumulative N(<z)")
axs[1].legend(fontsize=8)
axs[1].grid(alpha=0.3)
fig.suptitle(
    "Redshift distribution implied by GalaxyCountsMetricExtended / DepthLimitedNumGalMetric\n"
    "(power-law fits per z-bin on Padilla et al. mock catalogs, Awan et al. 2016)"
)
fig.tight_layout()
# save_fig(fig, "implicit_Nz_model")
save_fig_name = "implicit_Nz_model"
fig.savefig(save_fig_name + ".png", dpi=150, bbox_inches="tight")
fig.savefig(save_fig_name + ".pdf", bbox_inches="tight")

plt.show()

nz_table = pd.DataFrame(
    {
        "z_lo": z_lo,
        "z_hi": z_hi,
        "z_center": z_center,
        "a": [power_law_const_a[k] for k in zbin_keys],
        "b": [power_law_const_b[k] for k in zbin_keys],
        **{f"N [gal/deg2], m5={m5:.2f}": counts_per_zbin(m5) for m5 in DEPTHS_NZ},
    }
)
nz_table.to_csv(join(data_dir, "implicit_Nz_model.csv"), index=False)
display(nz_table.round(3))

## 4. Single-pixel experiment: what the two metrics return as a function of the coadded depth

The real metric objects are called through their `run(data_slice, slice_point)` method on a synthetic pixel: one visit per band, the i-band visit having `fiveSigmaDepth = m5` (a single visit gives a coadded depth equal to `m5`), and `E(B-V) = 0`. The u, g, r, z, y visits (depth 27) only serve to pass the 6-band coverage cut of `DepthLimitedNumGalMetric`. No OpSim database is needed.

- Left: the integrand (galaxies per deg² per magnitude, after completeness) for a pixel of coadded depth 26.2. The red part is what `GalaxyCountsMetricExtended` counts and `DepthLimitedNumGalMetric` does not (magnitudes beyond its hard limit).
- Middle: galaxies per deg² versus coadded depth. `DepthLimitedNumGalMetric` is masked below its depth cut, and above it the count grows more slowly than the `GalaxyCountsMetricExtended` one because the integral stops at the fixed limit.
- Right: ratio of the two.

In [ ]:
def synthetic_slice(m5_i, m5_other=27.0):
    # One visit per band; the i-band visit has depth m5_i (coadd of a single visit = its own depth)
    bands = ["u", "g", "r", "i", "z", "y"]
    rows = [(m5_i if b == BAND else m5_other, b) for b in bands]
    return np.array(rows, dtype=[("fiveSigmaDepth", "f8"), ("band", "U1")])


gc_ref = GalaxyCountsMetricExtended(nside=NSIDE, filter_band=BAND, **GC_KWARGS)
dl_ref = DepthLimitedNumGalMetric(nside=NSIDE, filter_band=BAND, lim_ebv=FIXED_LIM_EBV, **DL_KWARGS)
slice_point = {"ebv": 0.0}  # no dust: coadded depth = m5

pix_scale = gc_ref.scale  # deg^2 per pixel, as used inside the metrics
m_cut = dl_ref.galmetric.upper_mag_limit  # extended-source limit hard-wired in DepthLimitedNumGalMetric
print(f"GalaxyCountsMetricExtended: upper_mag_limit = {gc_ref.upper_mag_limit}")
print(
    f"DepthLimitedNumGalMetric  : upper_mag_limit = {m_cut:.2f} (= lim_mag_i_ptsrc - 0.7), "
    f"footprint depth cut = {dl_ref.eg_metric.depth_cut}, E(B-V) cut = {dl_ref.eg_metric.extinction_cut}, "
    f"n_filters = {dl_ref.eg_metric.n_filters}"
)

m5_grid = np.arange(24.0, 28.001, 0.05)
n_gc = np.array([gc_ref.run(synthetic_slice(m5), slice_point) for m5 in m5_grid], dtype=float)
n_dl = np.array([dl_ref.run(synthetic_slice(m5), slice_point) for m5 in m5_grid], dtype=float)
n_dl[n_dl == dl_ref.badval] = np.nan  # pixel rejected by the extragalactic cuts

m5_demo = 26.2
m_arr = np.linspace(20.0, 29.0, 500)
integrand = gc_ref._gal_count(m_arr, m5_demo) * normalization_constant  # gal / deg^2 / mag after completeness

fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].plot(m_arr, integrand, color="k")
axs[0].fill_between(
    m_arr,
    0,
    integrand,
    where=(m_arr <= m_cut),
    color="tab:blue",
    alpha=0.4,
    label=f"counted by both (m <= {m_cut:.1f})",
)
axs[0].fill_between(
    m_arr,
    0,
    integrand,
    where=(m_arr > m_cut),
    color="tab:red",
    alpha=0.4,
    label=f"counted only by GalaxyCountsMetricExtended (m > {m_cut:.1f})",
)
axs[0].axvline(m5_demo, color="gray", ls="--", label=f"coadded depth = {m5_demo}")
axs[0].set_xlabel("apparent i magnitude")
axs[0].set_ylabel("dN/dm x completeness [gal / deg² / mag]")
axs[0].set_title(f"Integrand for a pixel of coadded depth {m5_demo}")
axs[0].legend(fontsize=8, loc="upper left")

axs[1].semilogy(m5_grid, n_gc / pix_scale, label="GalaxyCountsMetricExtended (no cut, upper limit 32)")
axs[1].semilogy(m5_grid, n_dl / pix_scale, label=f"DepthLimitedNumGalMetric (upper limit {m_cut:.1f})")
axs[1].axvline(dl_ref.eg_metric.depth_cut, color="gray", ls="--", label="footprint depth cut of DepthLimited")
axs[1].set_xlabel("coadded i-band depth [mag]")
axs[1].set_ylabel("galaxies per deg²")
axs[1].legend(fontsize=8)
axs[1].grid(alpha=0.3)

axs[2].plot(m5_grid, n_dl / n_gc)
axs[2].set_xlabel("coadded i-band depth [mag]")
axs[2].set_ylabel("DepthLimited / GalaxyCountsExtended")
axs[2].grid(alpha=0.3)

fig.suptitle("Single-pixel experiment with the real metric objects (no dust)")
fig.tight_layout()
save_fig_name = join(figs_dir, "single_pixel_counts_vs_depth")
fig.savefig(save_fig_name + ".png", dpi=150, bbox_inches="tight")
fig.savefig(save_fig_name + ".pdf", bbox_inches="tight")
print("Saved:", save_fig_name + ".png/.pdf")
plt.show()

## 5. Sensitivity of the count to the point-to-extended offset (the PSF-related number)

The 0.7 mag between the point-source depth cut and the upper magnitude limit is hard-coded in `DepthLimitedNumGalMetric.__init__`. To see how much a different value would change the count, a `GalaxyCountsMetricExtended` is built with `upper_mag_limit = 26.0 - offset` (the same integral as `DepthLimitedNumGalMetric`, without footprint cuts) and run on synthetic pixels of a few coadded depths. The relative changes are given with respect to the 0.7 mag offset.

The second table is a **toy model** (not part of `rubin_sim`) of how such an offset depends on the seeing: a circular Gaussian galaxy of FWHM `theta_g` observed with a Gaussian PSF of FWHM `theta_p` is measured in an aperture whose area scales as `theta_p² + theta_g²`, so its limiting flux is brighter than the point-source one by `Δm = 1.25 log10(1 + (theta_g / theta_p)²)`.

In [ ]:
m5_values = [26.0, 26.2, 26.4]
offsets = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
count_cols = [f"gal/deg2, m5={m5}" for m5 in m5_values]

rows = []
for off in offsets:
    m_up = dl_pt - off
    metric = GalaxyCountsMetricExtended(
        nside=NSIDE, filter_band=BAND, **{**GC_KWARGS, "upper_mag_limit": m_up}
    )
    row = {"offset [mag]": off, "upper_mag_limit": round(m_up, 2)}
    for m5, col in zip(m5_values, count_cols):
        row[col] = metric.run(synthetic_slice(m5), slice_point) / pix_scale
    rows.append(row)
sens_df = pd.DataFrame(rows).set_index("offset [mag]")
rel_df = (sens_df[count_cols] / sens_df.loc[0.7, count_cols] - 1.0).add_suffix(" (rel. to 0.7)")
display(pd.concat([sens_df, rel_df], axis=1).round(3))

seeing = np.array([0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2])
toy = {f"galaxy FWHM = {g} arcsec": 1.25 * np.log10(1.0 + (g / seeing) ** 2) for g in [0.5, 1.0]}
toy_df = pd.DataFrame(toy, index=pd.Index(seeing, name="PSF FWHM [arcsec]"))
print("Toy model: point-source minus extended-source depth offset [mag]")
display(toy_df.round(2))

## 6. Helper functions

Same helpers as in notebook 01. Maps are numpy masked arrays (masked = pixel rejected or without data); `healpy` ignores the mask of a masked array, so masked pixels are converted to `hp.UNSEEN` (drawn in grey) before plotting.

In [ ]:
def as_healpy(m):
    # masked array -> plain array with hp.UNSEEN in the masked pixels (drawn in grey by healpy)
    return np.ma.filled(m, hp.UNSEEN)


def run_label(run_name):
    label = f"E(B-V) < {DUST_THRESH[run_name]:.3f}"
    return label + " (baseline)" if run_name.startswith("baseline") else label


def save_fig(fig, name):
    base = join(figs_dir, name)
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")


def bundle_to_masked(bundle):
    # MetricBundle.metric_values -> float masked array (MAF badval, NaN and inf are all masked)
    data = np.ma.getdata(bundle.metric_values).astype(float)
    mask = np.ma.getmaskarray(bundle.metric_values) | ~np.isfinite(data) | (data < -600.0)
    return np.ma.masked_array(data, mask=mask)


def save_map(path, m):
    np.savez_compressed(path, data=np.ma.getdata(m), mask=np.ma.getmaskarray(m))


def load_map(path):
    with np.load(path) as f:
        return np.ma.masked_array(f["data"], mask=f["mask"])


def diff_map(map_b, map_a, fill_masked=None):
    # Pixel-by-pixel map_b - map_a.
    # fill_masked=None: keep the difference only where BOTH maps are valid (masked elsewhere).
    # fill_masked=0.0 : a masked pixel counts as 0 (e.g. no galaxy counted there); the result is
    #                   masked only where BOTH maps are masked.
    mask_b, mask_a = np.ma.getmaskarray(map_b), np.ma.getmaskarray(map_a)
    b = np.ma.getdata(map_b).astype(float)
    a = np.ma.getdata(map_a).astype(float)
    if fill_masked is None:
        return np.ma.masked_array(b - a, mask=mask_b | mask_a)
    b = np.where(mask_b, fill_masked, b)
    a = np.where(mask_a, fill_masked, a)
    return np.ma.masked_array(b - a, mask=mask_b & mask_a)


def pair_stats(map_b, map_a, dmap, with_totals=False):
    # Footprint bookkeeping and statistics of one difference map
    vb, va = ~np.ma.getmaskarray(map_b), ~np.ma.getmaskarray(map_a)
    common, gained, lost = vb & va, vb & ~va, va & ~vb
    d = dmap.compressed()
    row = {
        "area_a_deg2": va.sum() * pix_area,
        "area_b_deg2": vb.sum() * pix_area,
        "area_gained_deg2": gained.sum() * pix_area,
        "area_lost_deg2": lost.sum() * pix_area,
        "diff_mean": d.mean() if d.size else np.nan,
        "diff_median": np.median(d) if d.size else np.nan,
        "diff_std": d.std() if d.size else np.nan,
        "diff_min": d.min() if d.size else np.nan,
        "diff_max": d.max() if d.size else np.nan,
    }
    if with_totals:
        b = np.ma.getdata(map_b).astype(float)
        a = np.ma.getdata(map_a).astype(float)
        row.update(
            {
                "total_a": a[va].sum(),
                "total_b": b[vb].sum(),
                "total_diff": b[vb].sum() - a[va].sum(),
                "from_common_pixels": (b[common] - a[common]).sum(),
                "from_gained_pixels": b[gained].sum(),
                "from_lost_pixels": -a[lost].sum(),
            }
        )
    return row


def per_run_summary(maps, with_sum=False):
    rows = []
    for run_name, dust in RUNS_INFO:
        v = maps[run_name].compressed()
        rows.append(
            {
                "run": run_name,
                "E(B-V) cut of the run": dust,
                "n_pixels": v.size,
                "area_deg2": v.size * pix_area,
                "mean": v.mean() if v.size else np.nan,
                "median": np.median(v) if v.size else np.nan,
                "std": v.std() if v.size else np.nan,
                "min": v.min() if v.size else np.nan,
                "max": v.max() if v.size else np.nan,
            }
        )
        if with_sum:
            rows[-1]["sum"] = v.sum()
    return pd.DataFrame(rows).set_index("run")


def plot_all_maps(maps, tag, unit, title, cmap="viridis"):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    vmin, vmax = np.percentile(pooled, 1), np.percentile(pooled, 99)
    fig = plt.figure(figsize=(15, 12))
    for i, (run_name, dust) in enumerate(RUNS_INFO, start=1):
        hp.mollview(
            as_healpy(maps[run_name]),
            fig=fig.number,
            sub=(3, 3, i),
            min=vmin,
            max=vmax,
            cmap=cmap,
            title=f"{run_name}\n{run_label(run_name)}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(f"{title} (common color scale: 1st-99th percentile of all runs)", fontsize=16, y=1.02)
    save_fig(fig, f"{tag}_healpix_allruns")
    plt.show()


def plot_overlay_hist(maps, tag, xlabel, title, nbins=80):
    pooled = np.concatenate([m.compressed() for m in maps.values()])
    bins = np.linspace(np.percentile(pooled, 0.1), np.percentile(pooled, 99.9), nbins + 1)
    colors = plt.cm.viridis(np.linspace(0.0, 0.95, len(RUNS_INFO)))
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    for (run_name, dust), c in zip(RUNS_INFO, colors):
        v = maps[run_name].compressed()
        w = np.full(v.size, pix_area)
        axs[0].hist(v, bins=bins, weights=w, histtype="step", color=c, label=run_label(run_name))
        axs[1].hist(
            v, bins=bins, weights=w, histtype="step", color=c, cumulative=-1, label=run_label(run_name)
        )
    axs[0].set_ylabel("Area per bin [deg²]")
    axs[1].set_ylabel("Area with value ≥ x [deg²]")
    for ax in axs:
        ax.set_xlabel(xlabel)
        ax.grid(alpha=0.3)
    axs[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    save_fig(fig, f"{tag}_histograms_allruns")
    plt.show()


def plot_diff_map(dmap, run_b, run_a, tag, unit, label):
    vlim = max(np.percentile(np.abs(dmap.compressed()), 99), 1e-6) if dmap.count() else 1.0
    fig = plt.figure(figsize=(8, 5))
    hp.mollview(
        as_healpy(dmap),
        fig=fig.number,
        min=-vlim,
        max=vlim,
        cmap="RdBu_r",
        title=f"{label} difference: {run_b}\nminus {run_a}",
        unit=unit,
    )
    hp.graticule()
    save_fig(fig, f"{tag}_diff_" + f"{run_b}_MINUS_{run_a}".replace(".", "_"))
    plt.show()
    return vlim


def plot_diff_mosaic(diff_maps, tag, unit, label):
    vlim = max(max(np.percentile(np.abs(d.compressed()), 99) for d in diff_maps.values() if d.count()), 1e-6)
    fig = plt.figure(figsize=(16, 10))
    for i, ((run_b, run_a), d) in enumerate(diff_maps.items(), start=1):
        hp.mollview(
            as_healpy(d),
            fig=fig.number,
            sub=(2, 3, i),
            min=-vlim,
            max=vlim,
            cmap="RdBu_r",
            title=f"{run_b}\nminus {run_a}",
            unit=unit,
            cbar=True,
        )
    fig.suptitle(
        f"{label}: differences between consecutive dust thresholds (common scale ±{vlim:.3g})",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_diff_allpairs_combined")
    plt.show()


def plot_footprint_change(maps, tag, label):
    cmap = ListedColormap(["tab:red", "lightgreen", "tab:blue"])
    fig = plt.figure(figsize=(16, 10))
    for i, (run_b, run_a) in enumerate(PAIRS, start=1):
        vb, va = ~np.ma.getmaskarray(maps[run_b]), ~np.ma.getmaskarray(maps[run_a])
        status = np.zeros(vb.size)
        status[vb & ~va] = 1.0
        status[va & ~vb] = -1.0
        status = np.ma.masked_array(status, mask=~(va | vb))
        hp.mollview(
            as_healpy(status),
            fig=fig.number,
            sub=(2, 3, i),
            min=-1,
            max=1,
            cmap=cmap,
            cbar=False,
            title=f"{run_b}\nminus {run_a}",
        )
    fig.suptitle(
        f"{label}: footprint change between consecutive dust thresholds "
        "(blue = pixel gained, red = pixel lost, green = valid in both, dark grey = valid in neither)",
        fontsize=16,
        y=1.02,
    )
    save_fig(fig, f"{tag}_footprint_change_combined")
    plt.show()


def plot_diff_histograms(diff_maps, tag, xlabel, label):
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for ax, ((run_b, run_a), d) in zip(axes.flat, diff_maps.items()):
        ax.hist(d.compressed(), bins=100, color="steelblue")
        ax.axvline(0, color="k", linewidth=0.8)
        ax.set_title(f"{run_b}\n- {run_a}", fontsize=9)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("N pixels")
    fig.suptitle(f"{label}: histograms of the consecutive differences")
    fig.tight_layout()
    save_fig(fig, f"{tag}_diff_histograms")
    plt.show()


def analyse_pairs(maps, tag, label, unit, xlabel, fill_masked=None, with_totals=False):
    # Difference maps (individual + mosaic), footprint change, histograms and table for all PAIRS
    diff_maps, rows = {}, []
    for run_b, run_a in PAIRS:
        print(f"=== {run_b}  -  {run_a} ===")
        d = diff_map(maps[run_b], maps[run_a], fill_masked=fill_masked)
        diff_maps[(run_b, run_a)] = d
        plot_diff_map(d, run_b, run_a, tag, unit, label)
        row = {"pair (E(B-V) thresholds)": f"{DUST_THRESH[run_b]:.3f} - {DUST_THRESH[run_a]:.3f}"}
        row.update(pair_stats(maps[run_b], maps[run_a], d, with_totals=with_totals))
        rows.append(row)
    plot_diff_mosaic(diff_maps, tag, unit, label)
    plot_footprint_change(maps, tag, label)
    plot_diff_histograms(diff_maps, tag, xlabel, label)
    stats = pd.DataFrame(rows).set_index("pair (E(B-V) thresholds)")
    stats.to_csv(join(data_dir, f"{tag}_diff_summary.csv"))
    print("Saved:", join(data_dir, f"{tag}_diff_summary.csv"))
    return diff_maps, stats

## 7. Compute (or reload) the two galaxy-count maps of each run

When both maps of a run have to be computed they are run in the same `MetricBundleGroup` (same query, same slicer, same dust map); in all cases they use exactly the same visits (same database and SQL constraint). A map is recomputed only if its cache file is missing: the `GalaxyCountsMetricExtended` maps do not depend on the E(B-V) cut and keep their cache, whereas the cache tag of the `DepthLimitedNumGalMetric` maps contains the `lim_ebv` used for the run, so those maps are recomputed once when the cut changes. The first execution is slow: each pixel requires numerical integrations (`scipy.integrate.quad`), for 196608 pixels per run. Results are cached in `data_02_GALCOUNTS/`.

In [ ]:
dustmap = maf_maps.DustMap(nside=NSIDE, interp=False)


def cache_path(kind, run_name):
    cfg = GC_CONFIG_TAG if kind == "GC" else dl_config_tag(run_name)
    tag = run_name.replace(".", "_")
    return join(data_dir, f"{kind}_{tag}_{cfg}.npz")


def load_or_run(run_name):
    paths = {kind: cache_path(kind, run_name) for kind in ("GC", "DL")}
    ebv_cut = metric_ebv_cut(run_name)
    todo = [kind for kind, p in paths.items() if FORCE_RECOMPUTE or not isfile(p)]
    out = {kind: load_map(paths[kind]) for kind in paths if kind not in todo}
    if not todo:
        print(f"[{run_name}] loaded cached maps (DepthLimitedNumGal: lim_ebv = {ebv_cut:.3f})")
        return out

    dbpath = get_db_path(run_name)
    print(f"[{run_name}] computing {todo} on {dbpath} (DepthLimitedNumGal: lim_ebv = {ebv_cut:.3f})")
    slicer = slicers.HealpixSlicer(nside=NSIDE, use_cache=False)
    gc_metric = GalaxyCountsMetricExtended(
        nside=NSIDE, filter_band=BAND, metric_name="GalaxyCountsMetricExtended", **GC_KWARGS
    )
    dl_metric = DepthLimitedNumGalMetric(
        nside=NSIDE, filter_band=BAND, metric_name="DepthLimitedNumGal", lim_ebv=ebv_cut, **DL_KWARGS
    )
    bundles = {
        "GC": mb.MetricBundle(
            gc_metric, slicer, SQL_CONSTRAINT, maps_list=[dustmap], run_name=run_name, info_label=INFO_LABEL
        ),
        "DL": mb.MetricBundle(
            dl_metric,
            slicer,
            SQL_CONSTRAINT,
            maps_list=[dustmap],
            run_name=run_name,
            info_label=f"{INFO_LABEL} ebv{ebv_cut:.3f}",
        ),
    }
    bundles = {
        kind: b for kind, b in bundles.items() if kind in todo
    }  # only the maps that are not cached yet
    group = mb.MetricBundleGroup(
        mb.make_bundles_dict_from_list(list(bundles.values())), dbpath, out_dir=data_dir, results_db=resultsDb
    )
    group.run_all()
    # (out already holds the maps loaded from the cache)
    for kind, bundle in bundles.items():
        out[kind] = bundle_to_masked(bundle)
        save_map(paths[kind], out[kind])
    return out


results = {run_name: load_or_run(run_name) for run_name, _ in RUNS_INFO}
gc_maps = {run_name: results[run_name]["GC"] for run_name in RUN_NAMES}
dl_maps = {run_name: results[run_name]["DL"] for run_name in RUN_NAMES}

## 8. Per-run summary

`area_deg2` is the area of the pixels where the metric is defined; `sum` is the **total number of galaxies** over the sky. For `GalaxyCountsMetricExtended` the defined pixels are all the pixels with at least one i-band visit (many of them with a small or zero count), for `DepthLimitedNumGalMetric` only the pixels that pass the extragalactic cuts.

In [ ]:
gc_summary = per_run_summary(gc_maps, with_sum=True).rename(columns={"sum": "total_galaxies"})
dl_summary = per_run_summary(dl_maps, with_sum=True).rename(columns={"sum": "total_galaxies"})
gc_summary.to_csv(join(data_dir, "GalaxyCountsMetricExtended_per_run_summary.csv"))
dl_summary.to_csv(join(data_dir, "DepthLimitedNumGal_per_run_summary.csv"))
print("GalaxyCountsMetricExtended (galaxies per pixel):")
display(gc_summary.round(1))
print("DepthLimitedNumGalMetric (galaxies per pixel):")
display(dl_summary.round(1))

### 8.1 Footprint effect and truncation effect

The total of `GalaxyCountsMetricExtended` is decomposed in two steps:
- **footprint effect** = (`GalaxyCountsMetricExtended` restricted to the pixels kept by `DepthLimitedNumGalMetric`) / (`GalaxyCountsMetricExtended` over all pixels): what the dust / 6-band / depth cuts remove;
- **truncation effect** = `DepthLimitedNumGalMetric` / (`GalaxyCountsMetricExtended` on the same pixels): what the hard 25.3 magnitude limit removes.

The last column checks that the depth-limited count never exceeds the unrestricted one, pixel by pixel.

In [ ]:
rows = []
for run_name, dust in RUNS_INFO:
    gc, dl = gc_maps[run_name], dl_maps[run_name]
    in_dl = ~np.ma.getmaskarray(dl)
    gc_data = np.ma.filled(gc, 0.0)
    dl_data = np.ma.filled(dl, 0.0)
    total_gc_all = gc_data.sum()
    total_gc_in_dl = gc_data[in_dl].sum()
    total_dl = dl_data.sum()
    rows.append(
        {
            "run": run_name,
            "E(B-V) cut of the run": dust,
            "GC, all pixels": total_gc_all,
            "GC, DL footprint only": total_gc_in_dl,
            "DL": total_dl,
            "footprint effect": total_gc_in_dl / total_gc_all,
            "truncation effect": total_dl / total_gc_in_dl,
            "DL <= GC everywhere": bool(np.all(dl_data[in_dl] <= gc_data[in_dl] + 1e-6)),
        }
    )
decomp_df = pd.DataFrame(rows).set_index("run")
decomp_df.to_csv(join(data_dir, "GC_vs_DL_decomposition.csv"))
decomp_df.round(3)

## 9. GalaxyCountsMetricExtended: maps and histograms

In [ ]:
plot_all_maps(
    gc_maps,
    "GalaxyCountsMetricExtended",
    "galaxies / pixel",
    "GalaxyCountsMetricExtended (i band, all z)",
)

In [ ]:
plot_overlay_hist(
    gc_maps,
    "GalaxyCountsMetricExtended",
    "galaxies / pixel",
    "GalaxyCountsMetricExtended: area-weighted histograms (left) and area with at least x galaxies / pixel (right)",
)

## 10. DepthLimitedNumGalMetric: maps and histograms

In [ ]:
plot_all_maps(
    dl_maps,
    "DepthLimitedNumGal",
    "galaxies / pixel",
    "DepthLimitedNumGalMetric (i band, all z)",
)

In [ ]:
plot_overlay_hist(
    dl_maps,
    "DepthLimitedNumGal",
    "galaxies / pixel",
    "DepthLimitedNumGalMetric: area-weighted histograms (left) and area with at least x galaxies / pixel (right)",
)

## 11. The two metrics side by side on one run

Reference run: `REF_RUN` (baseline). Left: `GalaxyCountsMetricExtended`. Middle: `DepthLimitedNumGalMetric`, same color scale. Right: their ratio on the pixels kept by `DepthLimitedNumGalMetric` (the truncation effect of Section 8.1, pixel by pixel). It depends on the coadded depth of the pixel and on nothing else.

In [ ]:
gc, dl = gc_maps[REF_RUN], dl_maps[REF_RUN]
ratio_mask = np.ma.getmaskarray(dl) | np.ma.getmaskarray(gc)
ratio = np.ma.masked_array(np.ma.getdata(dl) / np.maximum(np.ma.getdata(gc), 1e-12), mask=ratio_mask)
vmax = np.percentile(gc.compressed(), 99)

fig = plt.figure(figsize=(18, 5))
hp.mollview(
    as_healpy(gc),
    fig=fig.number,
    sub=(1, 3, 1),
    min=0,
    max=vmax,
    cmap="viridis",
    title="GalaxyCountsMetricExtended",
    unit="galaxies / pixel",
)
hp.mollview(
    as_healpy(dl),
    fig=fig.number,
    sub=(1, 3, 2),
    min=0,
    max=vmax,
    cmap="viridis",
    title="DepthLimitedNumGalMetric",
    unit="galaxies / pixel",
)
hp.mollview(
    as_healpy(ratio),
    fig=fig.number,
    sub=(1, 3, 3),
    min=0,
    max=1,
    cmap="magma",
    title="DepthLimited / GalaxyCountsExtended",
    unit="ratio",
)
fig.suptitle(f"{REF_RUN}: galaxy-count metrics side by side", fontsize=16, y=1.02)
save_fig(fig, "GC_vs_DL_side_by_side_" + REF_RUN.replace(".", "_"))
plt.show()

## 12. Differences between consecutive dust thresholds: GalaxyCountsMetricExtended

For each pair the map is `counts(run_b) - counts(run_a)` with `run_b` the run with the larger threshold. A masked pixel counts as 0 galaxies, so the map contains the pixels gained or lost with the footprint (grey = masked in both runs). In the table, `total_diff = from_common_pixels + from_gained_pixels + from_lost_pixels`: the change of the total number of galaxies split into a change of depth on the pixels valid in both runs, the galaxies of the pixels gained, and (negative) the galaxies of the pixels lost.

The `baseline-0.199` pair is a near-null test (almost identical nominal footprint): its differences give the scale of the scheduler-to-scheduler fluctuations.

In [ ]:
gc_diff_maps, gc_stats = analyse_pairs(
    gc_maps,
    "GalaxyCountsMetricExtended",
    "GalaxyCountsMetricExtended",
    "Δ galaxies / pixel",
    "Δ galaxies / pixel",
    fill_masked=0.0,
    with_totals=True,
)

In [ ]:
gc_stats.round(1)

## 13. Differences between consecutive dust thresholds: DepthLimitedNumGalMetric

Same construction. Here the footprint is defined by the metric itself (`ExgalM5WithCuts` with depth cut 26.0, see notebook 01), so the pixels gained or lost are those that cross the `E(B-V)`, 6-band and depth cuts from one run to the next. With `EBV_CUT_MODE = 'run'` the `E(B-V)` cut (`lim_ebv`) is the threshold of each run, so the gained pixels are expected to lie largely between the two thresholds of the pair.

In [ ]:
dl_diff_maps, dl_stats = analyse_pairs(
    dl_maps,
    "DepthLimitedNumGal",
    "DepthLimitedNumGalMetric",
    "Δ galaxies / pixel",
    "Δ galaxies / pixel",
    fill_masked=0.0,
    with_totals=True,
)

In [ ]:
dl_stats.round(1)

## 14. Total galaxies and footprint area versus the dust threshold

In [ ]:
dust_values = [d for _, d in RUNS_INFO]
tot_gc = [gc_maps[r].sum() for r in RUN_NAMES]
tot_dl = [dl_maps[r].sum() for r in RUN_NAMES]
area_dl = [dl_maps[r].count() * pix_area for r in RUN_NAMES]

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
axs[0].plot(dust_values, tot_gc, marker="o", color="tab:blue")
axs[0].set_ylabel("Total galaxies")
axs[0].set_title("GalaxyCountsMetricExtended")
axs[1].plot(dust_values, tot_dl, marker="o", color="tab:red")
axs[1].set_ylabel("Total galaxies")
axs[1].set_title("DepthLimitedNumGalMetric")
axs[2].plot(dust_values, area_dl, marker="o", color="tab:green")
axs[2].set_ylabel("Footprint area [deg²]")
axs[2].set_title("DepthLimitedNumGalMetric footprint")
for ax in axs:
    ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
    ax.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, "galaxy_counts_vs_dust_threshold")
plt.show()

### 14.1 Same plot, normalized to the baseline run

In [ ]:
i_ref = RUN_NAMES.index(REF_RUN)
tot_gc_n = np.array(tot_gc) / tot_gc[i_ref]
tot_dl_n = np.array(tot_dl) / tot_dl[i_ref]
area_dl_n = np.array(area_dl) / area_dl[i_ref]

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5))
axs[0].plot(dust_values, tot_gc_n, marker="o", color="tab:blue")
axs[0].set_ylabel("Total galaxies / baseline")
axs[0].set_title("GalaxyCountsMetricExtended")
axs[1].plot(dust_values, tot_dl_n, marker="o", color="tab:red")
axs[1].set_ylabel("Total galaxies / baseline")
axs[1].set_title("DepthLimitedNumGalMetric")
axs[2].plot(dust_values, area_dl_n, marker="o", color="tab:green")
axs[2].set_ylabel("Footprint area / baseline")
axs[2].set_title("DepthLimitedNumGalMetric footprint")
for ax in axs:
    ax.axhline(1.0, color="gray", ls="--", linewidth=0.8)
    ax.set_xlabel("E(B-V) threshold that defines the footprint of the run")
    ax.grid(alpha=0.3)
fig.suptitle("Same quantities, normalized to the baseline run", y=1.02)
fig.tight_layout()
save_fig(fig, "galaxy_counts_vs_dust_threshold_norm")
plt.show()

## 15. Notes for improving the metrics (PSF and weak lensing)

Facts read in the code and checked by the experiments above:

1. **Point-source depth used as galaxy depth.** In both metrics the completeness `0.5 * erfc(m - coaddm5)` is centred on the coadded **point-source** depth (`fiveSigmaDepth`, computed by the simulator with the seeing of each visit). There is no extended-source correction and no galaxy size or resolution criterion.
2. **A fixed 0.7 mag offset in `DepthLimitedNumGalMetric`.** It is the only PSF/size-like quantity, it is applied to the upper limit of the integral only (not to the completeness nor to the depth cut), and it does not depend on the actual seeing of the pixel. Section 5 gives the sensitivity of the count to this offset and the toy relation between the offset and the seeing.
3. **A hard cut instead of a depth-dependent one.** Since the upper limit is fixed at 25.3, the count of `DepthLimitedNumGalMetric` becomes much less sensitive to the coadded depth than the `GalaxyCountsMetricExtended` one when the pixel is well below the limit; Section 4 (middle and right panels) shows this directly.
4. **The footprint is a step function of the depth cut.** The `26.0` point-source cut in `DepthLimitedNumGalMetric` differs from the `25.9` used for the 3x2pt year-10 configuration (notebook 01, Section 10 gives the area difference).
5. **Hooks.** Both metrics read only `fiveSigmaDepth` and `band`. A PSF-aware version needs the seeing columns of the OpSim database (for example `seeingFwhmEff`) in the `col=[...]` list, then either an extended-source depth per visit before the coadd, or a per-pixel offset computed from the coadded seeing; a weak-lensing version could also require a resolution criterion (galaxy size versus PSF size) in the selection. The seeing maps of these runs are in `../07_variateEVmV/01_Seeing_HealpixMaps.ipynb`.


## Caveats

- Both metrics are simple models: power-law luminosity functions from mock catalogs per redshift bin, one colour correction between bands, a single `erfc` completeness. They are not a photometric selection function and not a forecast of a specific DESC sample; only differences between runs and between metric definitions are meaningful here.
- `redshift_bin='all'` sums all the power laws of the dictionary (see Section 3 for the number of bins in your installation).
- `nside` in the metric constructors only sets the deg² to pixel scale factor and must equal the slicer `nside` (128 here); otherwise all counts are off by a constant factor.
- A masked pixel counts as 0 galaxies in the difference maps and totals of Sections 12-14. Pixels with a count of 0 that are not masked (`GalaxyCountsMetricExtended` on shallow pixels) are valid pixels of the map and appear in its histogram.
- Two scheduler runs do not observe the same visit sequence even where their footprint is identical, so part of the pixel-to-pixel differences is scheduling noise (see the `baseline-0.199` pair).
- Every run is truncated to its first 10 years, so the numbers are not those of the full `baseline_v5.3.6_11yrs` simulation.
- The single-pixel experiments use synthetic pixels (no dust, one visit per band); they show how the metrics behave, not the depth actually reached in the simulations (see notebook 01 for that).
- `DepthLimitedNumGalMetric` uses `lim_ebv` = the E(B-V) threshold of each run (`EBV_CUT_MODE = 'run'`), whereas `GalaxyCountsMetricExtended` has no E(B-V) cut and counts every pixel with at least one i-band visit. The metric footprint is therefore expected to follow the scheduled WFD footprint, but the dust map used by MAF (`DustMap`, `nside = 128`, `interp=False`) is not necessarily the one used by the scheduler to build it, so pixels along the boundary may differ.


## References
- Awan, H. et al. 2016, ApJ 829, 50, "Testing LSST Dither Strategies for Survey Uniformity and Large-Scale Structure Systematics" - galaxy-count model implemented by `GalaxyCountsMetricExtended`. https://iopscience.iop.org/article/10.3847/0004-637X/829/1/50
- Lochner, M. et al. 2018, arXiv:1808.00006, "Optimizing LSST Observing Strategy for Dark Energy Science" - use of `DepthLimitedNumGalMetric`.
- LSST Science Collaboration 2009, "LSST Science Book", arXiv:0912.0201 (CFHTLS power law, eq. 3.7).
- `rubin_sim` sources: `maf_contrib/lss_obs_strategy/galaxy_counts_metric_extended.py`, `maf_contrib/lss_obs_strategy/constants_for_pipeline.py`, `maf_contrib/depth_limited_num_gal_metric.py`, `metrics/weak_lensing_systematics_metric.py` (`ExgalM5WithCuts`) - https://github.com/lsst/rubin_sim
- `../07_variateEVmV/01_FOMNv_HealpixDiff_ShrinkFPDust.ipynb` - presentation model; `01_compareExgalM5withCuts.ipynb` - companion notebook.
- `shrink_fp_dust_*.db` simulations: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/shrink_fp/
